# HMAC Encryption

**HMAC** (Hash-based Message Authentication Code) is a cryptographic function that maps an input string to a fixed-length hash value using a secret key. The process is **one-directional** and **irreversible** — given a hash value and even the secret key, it is computationally infeasible to recover the original input.

In this study, **HMAC-SHA256** is applied at the token level to encode address strings before matching, ensuring that raw address data is never directly exposed during the matching process.

Two tokenization strategies are considered: **1-gram** and **bigram**. The HMAC 1-gram (character level) process converts a Chinese address into an ordered sequence of secure, character-level hashes using a secret key, effectively creating a "blinded" version of the text. Under the bigram approach, each consecutive pair of characters — augmented with special start and end tokens — is hashed as a single unit, producing a set of hash values that captures local character sequences and preserves structural information about the address. 


In [ ]:
import hmac
import hashlib

def get_hmac_1grams(
    text, 
    secret_key="dia-123", 
    truncate_switch = False, 
    truncate_length = 12
    ):
    # 1. Normalization (Consider adding NFKC normalization for Chinese)
    clean_text = "".join(text.split()).strip().lower()
    key_bytes = secret_key.encode('utf-8')
    
    hmac_list = []
    
    for char in clean_text:
        h = hmac.new(key_bytes, char.encode('utf-8'), hashlib.sha256)
        if truncate_switch:
            hmac_list.append(h.hexdigest()[:truncate_length])
        else:
            hmac_list.append(h.hexdigest())
        
    return hmac_list

def get_hmac_2grams(
    text, 
    secret_key="dia-123",
    pad_switch=True, 
    truncate_switch=False, 
    truncate_length=12
    ):
    # 1. Normalization
    clean_text = "".join(text.split()).strip().lower()
    key_bytes = secret_key.encode('utf-8')
    
    # 2. Add special start/end tokens
    if pad_switch:
        padded_text = ["<S>"] + list(clean_text) + ["<E>"]
    else:
        padded_text = list(clean_text)
    
    hmac_list = []
    
    # 3. Slide over consecutive pairs
    for i in range(len(padded_text) - 1):
        bigram = padded_text[i] + padded_text[i + 1]
        h = hmac.new(key_bytes, bigram.encode('utf-8'), hashlib.sha256)
        if truncate_switch:
            hmac_list.append(h.hexdigest()[:truncate_length])
        else:
            hmac_list.append(h.hexdigest())
        
    return hmac_list

# Levenshtein Distance

`Levenshtein Distance` (often called Edit Distance) is a string metric that measures the minimum number of single-character edits required to change one word into another. These "edits" include:

Insertion: Adding a character.

Deletion: Removing a character.

Substitution: Replacing one character with another.

`Levenshtein Similarity` = 1 - `Levenshtein Distance`/max string lentgh

In [12]:
def levenshtein_hmac(list_a, list_b):
    """
    Calculates the Levenshtein distance between two lists of HMAC hashes.
    """
    size_a = len(list_a)
    size_b = len(list_b)
    
    # Initialize the distance matrix
    # Rows represent list_a, Columns represent list_b
    matrix = [[0] * (size_b + 1) for _ in range(size_a + 1)]

    # Fill the first row and column (base cases for empty strings)
    for i in range(size_a + 1):
        matrix[i][0] = i
    for j in range(size_b + 1):
        matrix[0][j] = j

    # Compute the edit distance
    for i in range(1, size_a + 1):
        for j in range(1, size_b + 1):
            # If the hashes match, the cost is 0
            if list_a[i-1] == list_b[j-1]:
                cost = 0
            else:
                cost = 1
            
            matrix[i][j] = min(
                matrix[i-1][j] + 1,      # Deletion
                matrix[i][j-1] + 1,      # Insertion
                matrix[i-1][j-1] + cost  # Substitution
            )

    return matrix[size_a][size_b]

# --- Example Usage ---
secret = "dia-123"

address_a = "北京市朝阳区花家地北里1号楼1501"
address_b = "北京市朝阳区花家地北里11号楼701"

hashes_a = get_hmac_1grams(address_a, secret, truncate_switch=True, truncate_length=12)
hashes_b = get_hmac_1grams(address_b, secret, truncate_switch=True, truncate_length=12)

distance = levenshtein_hmac(hashes_a, hashes_b)
similarity = 1 - distance/max(len(address_a), len(address_b))

print(f"Levenshtein Distance: {distance}", f"Levenshtein Similarity: {similarity:.2f}")

Levenshtein Distance: 3 Levenshtein Similarity: 0.83


# Jaro Similarity

`Jaro Similarity` is a string metric designed primarily for short strings like names and addresses, focusing specifically on character matches and transpositions.

$$sim_j = \begin{cases} 0 & \text{if } m = 0 \\ \frac{1}{3} \left( \frac{m}{|s_1|} + \frac{m}{|s_2|} + \frac{m-t}{m} \right) & \text{otherwise} \end{cases}$$


In [14]:
def jaro_similarity_hmac(list_a, list_b):
    len_a, len_b = len(list_a), len(list_b)
    if len_a == 0 or len_b == 0:
        return 0.0

    # Maximum distance for a "match"
    match_distance = max(len_a, len_b) // 2 - 1
    
    matches_a = [False] * len_a
    matches_b = [False] * len_b
    
    m = 0
    # Find matching characters within the allowed distance
    for i in range(len_a):
        start = max(0, i - match_distance)
        end = min(i + match_distance + 1, len_b)
        for j in range(start, end):
            if not matches_b[j] and list_a[i] == list_b[j]:
                matches_a[i] = True
                matches_b[j] = True
                m += 1
                break
                
    if m == 0:
        return 0.0

    # Count transpositions
    t = 0
    k = 0
    for i in range(len_a):
        if matches_a[i]:
            while not matches_b[k]:
                k += 1
            if list_a[i] != list_b[k]:
                t += 1
            k += 1
    
    transpositions = t // 2
    
    # Calculate Jaro Score
    return (1/3) * (m/len_a + m/len_b + (m - transpositions)/m)

similarity = jaro_similarity_hmac(hashes_a, hashes_b)
print(f"Jaro Similarity: {similarity:.4f}")

Jaro Similarity: 0.9434


## Jaro-Winkler

Enhancement to Jaro similarity that gives extra credit to strings that match at the beginning.

In [15]:
def jaro_winkler_hmac(list_a, list_b, p=0.1):
    # 1. Get the base Jaro similarity
    j_score = jaro_similarity_hmac(list_a, list_b)
    
    # 2. Calculate prefix length (l) up to 4 characters
    l = 0
    max_prefix = min(len(list_a), len(list_b), 4)
    
    for i in range(max_prefix):
        if list_a[i] == list_b[i]:
            l += 1
        else:
            break
            
    # 3. Apply the Winkler modification
    w_score = j_score + (l * p * (1 - j_score))
    
    return w_score


# Example Usage

w_score = jaro_winkler_hmac(hashes_a, hashes_b)
print(f"Jaro-Winkler Similarity: {w_score:.4f}")

Jaro-Winkler Similarity: 0.9660


# Qgram

The **Multiset Q-gram** approach breaks a sequence into overlapping tokens of length $q$ and stores them in a "bag" (Counter) to preserve the exact frequency of each segment. The similarity is calculated by summing the minimum occurrences of shared tokens and dividing by a chosen denominator, such as the length of the longer, shorter, or average sequence.

In [17]:
from collections import Counter

def get_qgrams_from_hash_list(hash_list, q=2, padded=True):
    """
    Groups a list of 1-gram hashes into q-gram tuples.
    Input: ['hash1', 'hash2', 'hash3']
    Output (q=2, padded=True): [('#', 'hash1'), ('hash1', 'hash2'), ('hash2', 'hash3'), ('hash3', '#')]
    """
    processed_list = list(hash_list)
    
    # 1. Apply padding using a unique sentinel for the 'hash' space
    if padded:
        # We use a non-hex string or a specific sentinel to represent the pad
        pad_token = ["#PAD#"] * (q - 1)
        processed_list = pad_token + processed_list + pad_token
    
    qgram_list = []
    
    # 2. Sliding window over the hash IDs
    for i in range(len(processed_list) - q + 1):
        # We store as a tuple because tuples are hashable (can be used in Counter/Sets)
        qgram_tuple = tuple(processed_list[i:i+q])
        qgram_list.append(qgram_tuple)
        
    return qgram_list

# --- Example Usage ---
q_grams = get_qgrams_from_hash_list(hashes_a, q=3, padded=True)

for gram in q_grams:
    print(gram)

('#PAD#', '#PAD#', '7e94ad4d1f39')
('#PAD#', '7e94ad4d1f39', '62256d235ac8')
('7e94ad4d1f39', '62256d235ac8', '0da2fa1103d9')
('62256d235ac8', '0da2fa1103d9', 'b3e427c17da2')
('0da2fa1103d9', 'b3e427c17da2', '20b091c52910')
('b3e427c17da2', '20b091c52910', '643c6722caad')
('20b091c52910', '643c6722caad', '1b557f28e7ad')
('643c6722caad', '1b557f28e7ad', 'fd4543917a91')
('1b557f28e7ad', 'fd4543917a91', 'e9d0d0a99e53')
('fd4543917a91', 'e9d0d0a99e53', '7e94ad4d1f39')
('e9d0d0a99e53', '7e94ad4d1f39', 'bfd6b5599cfc')
('7e94ad4d1f39', 'bfd6b5599cfc', 'bfabc74b60a7')
('bfd6b5599cfc', 'bfabc74b60a7', 'efb726339cce')
('bfabc74b60a7', 'efb726339cce', 'e20390904148')
('efb726339cce', 'e20390904148', 'bfabc74b60a7')
('e20390904148', 'bfabc74b60a7', '40ae85112a3e')
('bfabc74b60a7', '40ae85112a3e', 'c89d80ebd2fa')
('40ae85112a3e', 'c89d80ebd2fa', 'bfabc74b60a7')
('c89d80ebd2fa', 'bfabc74b60a7', '#PAD#')
('bfabc74b60a7', '#PAD#', '#PAD#')


In [43]:
from collections import Counter

def calculate_qgram_similarity(qgram_list_a, qgram_list_b, denominator_type="average"):
    """
    Calculates similarity between two pre-generated q-gram lists using multiset logic.
    
    Parameters:
    - qgram_list_a, qgram_list_b: Lists of q-gram hashes or tuples.
    - denominator_type: 'longer', 'shorter', or 'average' (Sørensen–Dice).
    """
    if not qgram_list_a or not qgram_list_b:
        return 0.0

    # 1. Convert lists to multisets (Counters) to track frequencies
    counter_a = Counter(qgram_list_a)
    counter_b = Counter(qgram_list_b)
    
    # 2. Find the size of the intersection (sum of min counts for each shared q-gram)
    # The & operator on Counters returns the intersection multiset
    intersection_size = sum((counter_a & counter_b).values())
    
    len_a = len(qgram_list_a)
    len_b = len(qgram_list_b)

    # 3. Apply the chosen denominator logic
    if denominator_type == "longer":
        denominator = max(len_a, len_b)
    elif denominator_type == "shorter":
        denominator = min(len_a, len_b)
    elif denominator_type == "average":
        # This is the Sørensen–Dice denominator
        denominator = (len_a + len_b) / 2
    else:
        raise ValueError("Invalid denominator_type. Choose 'longer', 'shorter', or 'average'.")

    return intersection_size / denominator

# Example Usage

address_a = "北京市朝阳区花家地北里22号楼107"
address_b = "北京市朝阳区花家地北里33号楼401"

hashes_a = get_hmac_1grams(address_a, secret, truncate_switch=True, truncate_length=12)
hashes_b = get_hmac_1grams(address_b, secret, truncate_switch=True, truncate_length=12)


qgram_list_a = get_qgrams_from_hash_list(hashes_a, q=2, padded=True)
qgram_list_b = get_qgrams_from_hash_list(hashes_b, q=2, padded=True)

score = calculate_qgram_similarity(qgram_list_a, qgram_list_b, denominator_type="average")
print(f"Similarity Score: {score:.4f}")

Similarity Score: 0.6316


# Jaccard Distance

In [44]:
def calculate_jaccard(set1, set2):
    """Calculates Jaccard Similarity between two sets of hashes."""
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return intersection / union if union > 0 else 0

similarity = calculate_jaccard(set(hashes_a), set(hashes_b))

print(f"Similarity Score: {similarity:.2f}")

Similarity Score: 0.78


# Parse Address

In [10]:
import hanlp

han_nlp = hanlp.load(
    hanlp.pretrained.mtl.CLOSE_TOK_POS_NER_SRL_DEP_SDP_CON_ELECTRA_BASE_ZH,
    tokenizer_kwargs={'fix_mistral_regex': True})



c:\Users\gongz\experiment\HMAC_addresses\.venv\Lib\site-packages\torch\cuda\__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
c:\Users\gongz\experiment\HMAC_addresses\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [25]:
address = "济南莱芜区文化路一号"

results = han_nlp(address, tasks='ner')
print(results)

{
  "tok/fine": [
    "济南",
    "莱芜区",
    "文化路",
    "一",
    "号"
  ],
  "ner/msra": [
    ["济南", "LOCATION", 0, 1],
    ["莱芜区", "LOCATION", 1, 2],
    ["文化路一号", "LOCATION", 2, 5]
  ]
}


In [26]:
import addressparser

df = addressparser.transform([address])
df

,省,市,区,地名
0,山东省,济南市,莱芜区,文化路一号


In [ ]:
address = "上海市西安路115号花园小区1号楼601"
df = addressparser.transform([address])
df 

,省,市,区,地名
0,上海市,上海市,,西安路115号花园小区1号楼601


# NER: fine-tuned BERT

In [34]:
from transformers import pipeline
import re

ner = pipeline(
    "token-classification",
    model="jiaqianjing/chinese-address-ner",
)

LABEL_MAP = {
    0:  ("other",     "O"),
    1:  ("province",  "B"),
    2:  ("province",  "I"),
    3:  ("city",      "B"),
    4:  ("city",      "I"),
    5:  ("district",  "B"),
    6:  ("district",  "I"),
    7:  ("street",    "B"),
    8:  ("street",    "I"),
    9:  ("community", "B"),
    10: ("community", "I"),
    11: ("unknown",   "B"),
    12: ("unknown",   "I"),
    13: ("road",      "B"),  # also restarts for building name
    14: ("road",      "I"),
}

def clean_token(word: str) -> str:
    """Remove BERT WordPiece ## artifacts."""
    return word.replace("##", "")

def extract_road_segments(spans: list[str]) -> dict:
    result = {}
    remainder_parts = []

    for span in spans:
        road_match = re.match(r"(.+?[路街道巷弄])(\d+)号?(.*)", span)
        if road_match:
            result["road"] = road_match.group(1)
            if road_match.group(2):
                result["road_num"] = road_match.group(2)
            if road_match.group(3):
                remainder_parts.append(road_match.group(3))
        else:
            remainder_parts.append(span)

    if remainder_parts:
        result["unit"] = "".join(remainder_parts)

    return result

def parse_address(address: str) -> dict:
    tokens = ner(address)
    parsed = {}
    current_field = None
    current_text = ""
    road_spans = []
    unit_text = ""
    last_was_road = False

    def save():
        nonlocal current_field, current_text
        if not current_field or not current_text:
            return
        if current_field == "road":
            road_spans.append(current_text)
        else:
            parsed[current_field] = parsed.get(current_field, "") + current_text
        current_field, current_text = None, ""

    for token in tokens:
        idx = int(token["entity"].split("_")[1])
        field, bio = LABEL_MAP[idx]
        word = clean_token(token["word"])

        if field == "other":
            if current_field:
                save()
            if last_was_road:
                unit_text += word  # accumulate LABEL_0 after LABEL_14 as unit
            continue

        last_was_road = (field == "road")

        if bio == "B":
            save()
            current_field = field
            current_text = word
        else:
            current_text += word

    save()

    if road_spans:
        parsed.update(extract_road_segments(road_spans))

    if unit_text:
        parsed["unit"] = parsed.get("unit", "") + unit_text

    return parsed


Device set to use cpu


In [ ]:
# Test
test_cases = [
    "广东省深圳市南山区科技园路88号",
    "上海市浦东新区陆家嘴环路1000号",
    "四川省成都市锦江区春熙路步行街18号3楼",
    "浙江省杭州市西湖区文三路477号天苑大厦5楼501室",
    "北京市海淀区西北旺东路10号院马连洼街道西北旺社区",
]

parsed_data = [parse_address(add) for add in test_cases]
for parsed in parsed_data:
    print(parsed)

{'province': '广东省', 'city': '深圳市', 'district': '南山区', 'road': '科技园路', 'road_num': '88'}
{'province': '上海市', 'district': '浦东新区', 'road': '陆家嘴环路', 'road_num': '1000'}
{'province': '四川省', 'city': '成都市', 'district': '锦江区', 'road': '春熙路步行街', 'road_num': '18', 'unit': '3楼'}
{'province': '浙江省', 'city': '杭州市', 'district': '西湖区', 'road': '文三路', 'road_num': '477', 'unit': '天苑大厦5楼501室'}
{'province': '北京市', 'district': '海淀区', 'street': '马连洼街道', 'community': '西北旺社区', 'road': '西北旺东路', 'road_num': '10', 'unit': '院'}


c:\Users\gongz\experiment\HMAC_addresses\.venv\Lib\site-packages\torch\nn\modules\module.py:1787: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
